# Query Rewriting — LLM Enriches Your Query Before Search

**Problem:** Users write vague, incomplete queries.
- "how did revolution affect farming?" — Which revolution? What farming?

**Solution:** Send the query to an LLM first. Let it add specificity, keywords, and context.

**Pipeline:** User Query → LLM Rewrite → Semantic Search → Results

In [ ]:
# Simulating query rewriting (no LLM needed for the demo)

rewrites = {
    "how did revolution affect farming?": 
        "How did the Industrial Revolution (1760-1840) impact agricultural practices, "
        "crop production, rural labor, and the enclosure movement in Britain?",
    
    "tall buildings in New York":
        "Skyscrapers and high-rise buildings in New York City including the Empire State Building, "
        "Chrysler Building, One World Trade Center, and other notable tall structures in Manhattan",
    
    "medicine for sugar":
        "Medications and pharmaceutical treatments for diabetes mellitus including metformin, "
        "insulin, GLP-1 receptor agonists, SGLT2 inhibitors, and sulfonylureas for blood sugar control",
}

for original, rewritten in rewrites.items():
    print(f"Original:  \"{original}\"")
    print(f"Rewritten: \"{rewritten}\"")
    print(f"Added keywords: {len(rewritten.split()) - len(original.split())} new words")
    print()

## Why Does This Help?

More keywords = better BM25 matching. More specific meaning = better semantic matching.

Let's prove it with embeddings:

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Target document we want to find
target = (
    "The Industrial Revolution (1760-1840) transformed manufacturing from hand production "
    "to machine-based processes. Agricultural changes included the seed drill, crop rotation, "
    "and enclosure of common lands, which displaced rural workers into factory towns."
)

original_query = "how did revolution affect farming?"
rewritten_query = rewrites[original_query]

target_emb = model.encode(target)
original_emb = model.encode(original_query)
rewritten_emb = model.encode(rewritten_query)

sim_original = cosine_sim(original_emb, target_emb)
sim_rewritten = cosine_sim(rewritten_emb, target_emb)
improvement = ((sim_rewritten - sim_original) / sim_original) * 100

print(f"Target doc: \"{target[:80]}...\"")
print(f"\nOriginal query similarity:  {sim_original:.4f}")
print(f"Rewritten query similarity: {sim_rewritten:.4f}")
print(f"Improvement: +{improvement:.1f}%")

## The Rewrite Prompt

The quality of rewriting depends entirely on the prompt:

```
You are a search query optimizer. Rewrite the user's query to be more
specific, detailed, and search-friendly.

Rules:
- Keep the original intent
- Add specific terms, synonyms, and related concepts
- Expand abbreviations and vague references
- Output ONLY the rewritten query
```

## When Does Query Rewriting Help vs Hurt?

| Scenario | Helps? | Why |
|----------|--------|-----|
| Vague query: "revolution and farming" | Yes | Adds specificity |
| Already specific: "metformin 500mg side effects" | No | Rewriting might add noise |
| Ambiguous: "apple" (fruit or company?) | Depends | LLM might guess wrong |
| Short query: "TSMC" | Yes | Expands to full name + context |

## Key Takeaways

1. Query rewriting is a **pre-retrieval** technique — it improves the query, not the search
2. The rewritten query goes through normal retrieval (BM25, Semantic, etc.)
3. Adds **latency** (LLM call = 200-500ms) but often **improves recall significantly**
4. Works best on **vague, short, or ambiguous queries**